<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 강화학습·에이전트 · 03. Dueling DQN

## Dueling Network Architectures for Deep Reinforcement Learning

- **원 논문:** [Dueling Network Architectures for Deep Reinforcement Learning](https://proceedings.mlr.press/v48/wangf16.html)
- **저자 / 발표:** Ziyu Wang et al. · ICML (2016)
- **난이도 / 예상 시간:** 중급 · 약 50분
- **선수 지식:** Q-learning, value/advantage decomposition
- **로컬 학습 데이터:** `data/field_curriculum/gridworld.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** V와 A stream을 mean-centered aggregation으로 결합해 로컬 optimal Q table을 근사한다.

**축소하거나 생략한 부분:** 논문은 Atari DQN backbone을 두 stream으로 분리한다. 축소판은 architecture/identifiability를 tabular-size MLP로 격리한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3, Eq. (7) | naive V+A decomposition | 식별 불가능한 상수 이동 문제를 관찰한다. |
| §3, Eq. (8)–(9) | max/mean-centered aggregation | 실험에 쓰인 Q=V+A-mean(A)를 구현한다. |
| §4.1, corridor policy-evaluation experiment | action-redundancy interpretation | 상태가치와 action-specific advantage를 따로 시각화한다. |

## 핵심 재현

        Eq. (9) $Q(s,a)=V(s)+A(s,a)-\frac1{|A|}\sum_{a'}A(s,a')$를 구현합니다.
        평균 advantage가 0이므로 V가 Q의 action 평균으로 식별됩니다.

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(431)
np.random.seed(431)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# Tiny scalar/table loops may be faster on CPU even when a GPU exists;
# AI_LAB_DEVICE=cpu remains the low-overhead override.
DATA_PATH = Path("data/field_curriculum/gridworld.json")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
config = json.loads(DATA_PATH.read_text(encoding="utf-8"))
required = {
    "seed",
    "size",
    "start_state",
    "terminal_state",
    "gamma",
    "transitions",
    "bandit_means",
    "offline_action_noise",
}
assert required.issubset(config), f"JSON 키 불일치: {sorted(config)}"
rows = config["transitions"]
states = torch.tensor([int(r["state"]) for r in rows], dtype=torch.long)
actions = torch.tensor([int(r["action"]) for r in rows], dtype=torch.long)
rewards = torch.tensor([float(r["reward"]) for r in rows], dtype=torch.float32)
next_states = torch.tensor([int(r["next_state"]) for r in rows], dtype=torch.long)
dones = torch.tensor([float(r["done"]) for r in rows], dtype=torch.float32)
n_states = int(max(states.max(), next_states.max()).item()) + 1
n_actions = int(actions.max().item()) + 1
gamma = float(config["gamma"])
start_state, terminal_state = int(config["start_state"]), int(config["terminal_state"])
transition_table = {
    (int(r["state"]), int(r["action"])): (
        int(r["next_state"]),
        float(r["reward"]),
        bool(r["done"]),
    )
    for r in rows
}


def encode(s):
    return F.one_hot(torch.as_tensor(s, dtype=torch.long), n_states).float()


def encode_device(s):
    return encode(s).to(DEVICE)


def env_step(state, action):
    return transition_table[(int(state), int(action))]


def value_iteration(iterations=300):
    q = torch.zeros(n_states, n_actions)
    for _ in range(iterations):
        old = q.clone()
        for (s, a), (ns, r, done) in transition_table.items():
            q[s, a] = r if done else r + gamma * old[ns].max()
    return q


q_star = value_iteration()
assert torch.isfinite(q_star).all() and len(rows) > 0
print(ACCELERATOR.summary())
print(
    "RL environment, replay metadata, and tree search stay on CPU; "
    "neural batches use DEVICE."
)
print(
    f"local MDP: states={n_states}, actions={n_actions}, "
    f"transitions={len(rows)}, gamma={gamma}"
)

## 포트폴리오 구현 설계: 수식 → 에이전트 책임 → 검증

논문의 핵심 update를 실행 가능한 에이전트의 `forward`, `loss`, `update`,
`evaluate`로 나눕니다. 아래 식의 선택·평가·gradient 경계를 메서드 본문에서
한 줄씩 추적하세요.

$$
Q(s,a)=V(s)+A(s,a)-\frac{1}{|\mathcal A|}\sum_{a'}A(s,a')
$$

- **기호와 역할:** $V(s)$는 상태가치 scalar, $A(s,a)$는 action advantage, 평균 제거는 두 stream의 식별 가능성을 만듭니다.
- **shape/state 계약:** `one-hot state [B, S] → V [B, 1], A/Q [B, A]`
- **논문 위치:** `§3, Eq. (7)`의 **naive V+A decomposition**
- **코드 Task:** shared feature, value/advantage heads, centered aggregation과 Q 평가를 구현합니다.
- **학습·평가 흐름:** state encode → V/A streams → mean-center → Q loss → stream 해석
- **원문 대비 한계:** 논문은 Atari DQN backbone을 두 stream으로 분리한다. 축소판은 architecture/identifiability를 tabular-size MLP로 격리한다.
- **구현 이유:** target, gradient, optimizer 책임을 Agent 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 Bellman target, policy objective, search score 같은 핵심 계산을
불투명한 trainer 뒤에 감추지 않습니다. 실습본에서는 공개 API를 유지하면서 TODO를
채우고, 각 메서드의 입력 shape와 gradient가 끊기는 위치를 설명하세요.

In [ ]:
class DuelingValueAgent(nn.Module):
    """Dueling DQN의 핵심 학습·평가 경로. 입출력 계약: one-hot state [B, S] → V [B, 1], A/Q [B,
    A]."""

    def __init__(self, network, optimizer):
        """network, optimizer, target 또는 table 상태를 명시적으로 저장한다."""
        super().__init__()
        self.network = network
        self.optimizer = optimizer

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        return {"state_count": n_states, "action_count": n_actions, "centering": "mean"}

    def forward(self, state):
        """policy/value/Q 순전파를 계산한다. shape 계약: one-hot state [B, S] → V [B, 1], A/Q [B,
        A]."""
        encoded_state = encode_device(state)
        value, advantage = self.network.streams(encoded_state)
        centered_advantage = advantage - advantage.mean(dim=1, keepdim=True)
        q_value = value + centered_advantage
        return (q_value, value, centered_advantage)

    def loss(self, state, target_q):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        predicted_q, _, _ = self.forward(state)
        return F.mse_loss(predicted_q, target_q)

    def update(self, state, target_q):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        objective = self.loss(state, target_q)
        self.optimizer.zero_grad()
        objective.backward()
        self.optimizer.step()
        return float(objective.detach())

    def evaluate(self, state, target_q):
        """no_grad 경로에서 정책 또는 value 품질 지표 dict를 반환한다."""
        with torch.no_grad():
            predicted_q, value, centered_advantage = self.forward(state)
            mse = F.mse_loss(predicted_q, target_q)
            centering_error = centered_advantage.mean(dim=1).abs().max()
        return {
            "q_mse": float(mse),
            "centering_error": float(centering_error),
            "value_shape": tuple(value.shape),
        }

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `§3, Eq. (8)–(9)` — **max/mean-centered aggregation**
- **구현 이유:** 실험에 쓰인 Q=V+A-mean(A)를 구현한다.
- **읽을 코드:** 아래 `implementation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
class DuelingQ(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature = nn.Sequential(nn.Linear(n_states, 32), nn.ReLU())
        self.value = nn.Linear(32, 1)
        self.advantage = nn.Linear(32, n_actions)

    def streams(self, x):
        h = self.feature(x)
        return self.value(h), self.advantage(h)

    def forward(self, x):
        v, a = self.streams(x)
        return v + a - a.mean(1, keepdim=True)


model = DuelingQ().to(DEVICE)
assert model(encode_device(torch.arange(n_states))).shape == (
    n_states,
    n_actions,
)

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `§3, Eq. (7)` — **naive V+A decomposition**
- **구현 이유:** 식별 불가능한 상수 이동 문제를 관찰한다.
- **읽을 코드:** 아래 `training` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=0.015)
all_states = encode_device(torch.arange(n_states))
q_ref = q_star.to(DEVICE)
history = []
with torch.no_grad():
    initial = float(F.mse_loss(model(all_states), q_ref))

portfolio_agent = DuelingValueAgent(model, opt)
state_ids = torch.arange(n_states)
initial_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in model.parameters()]
)
for _ in range(220):
    history.append(portfolio_agent.update(state_ids, q_ref))
final_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in model.parameters()]
)
parameter_delta = float((final_parameters - initial_parameters).abs().sum())
assert history[-1] < 0.05 * initial and parameter_delta > 0.0

### 단계 4. 평가·시각화

- **논문의 어느 부분인가:** `§4.1, corridor policy-evaluation experiment` — **action-redundancy interpretation**
- **구현 이유:** 상태가치와 action-specific advantage를 따로 시각화한다.
- **읽을 코드:** 아래 `evaluation` 셀에서 state/action tensor의 shape, gradient가
  흐르는 경로와 target 또는 old-policy가 고정되는 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
portfolio_metrics = portfolio_agent.evaluate(torch.arange(n_states), q_ref)
assert portfolio_metrics

with torch.no_grad():
    v, a = model.streams(all_states)
    centered = a - a.mean(1, keepdim=True)
    learned = v + centered
    mse = float(F.mse_loss(learned, q_ref))
assert (
    torch.allclose(centered.mean(1), torch.zeros_like(centered.mean(1)), atol=1e-6)
    and mse < 0.02
)
print(f"Q-table MSE={mse:.6f}")
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].plot(history)
axes[0].set_title("fit loss")
axes[1].plot(v.detach().cpu()[:, 0])
axes[1].set_title("V(s)")
axes[2].imshow(centered.detach().cpu().T, aspect="auto", cmap="coolwarm")
axes[2].set_title("centered A(s,a)")
fig.tight_layout()
plt.show()

V에 c를 더하고 모든 A에서 c를 빼도 Q가 같다는 자유도를 ΣA=0 제약으로 제거합니다. 따라서 V는 action 평균 Q가 됩니다.

### 단계 5. 통합 Agent가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `§3, Eq. (7)`의 **naive V+A decomposition**
- **핵심 식:**

  $$
  Q(s,a)=V(s)+A(s,a)-\frac{1}{|\mathcal A|}\sum_{a'}A(s,a')
  $$

- **입출력 shape:** `one-hot state [B, S] → V [B, 1], A/Q [B, A]`
- **구현 이유:** 앞 셀은 target, objective, search를 작은 연산으로
  분해해 확인하고, 이 셀은 같은 구성 요소를
  `DuelingValueAgent`에 주입해 최종 학습·평가 경로를
  하나로 묶습니다. 식별 불가능한 상수 이동 문제를 관찰한다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 정책·가치 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

In [ ]:
config_contract = portfolio_agent.config()
numeric_metrics = [
    float(value)
    for value in portfolio_metrics.values()
    if isinstance(value, (int, float))
]
assert config_contract and len(history) > 0
assert np.isfinite(np.asarray(history, dtype=float)).all()
assert parameter_delta > 0.0
assert numeric_metrics and np.isfinite(numeric_metrics).all()
print(
    {
        "class": type(portfolio_agent).__name__,
        "steps": len(history),
        "parameter_delta": parameter_delta,
        "metrics": portfolio_metrics,
    }
)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 논문은 Atari DQN backbone을 두 stream으로 분리한다. 축소판은 architecture/identifiability를 tabular-size MLP로 격리한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.